[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SekyungHan/ai-power-textbook-labs/blob/master/solutions/ch01_neural_network_basics.ipynb)

# Ch 1: 신경망의 본질 — IEEE 14-bus MLP 전압 예측 (정답)

# Ch01: 신경망의 본질 — IEEE 14-bus MLP 전압 예측 (Solution)

이 실습에서는 **pandapower**를 사용하여 IEEE 14-bus 전력 시스템의 부하 시나리오를 생성하고,
간단한 **다층 퍼셉트론(MLP)**으로 버스 전압을 예측하는 모델을 학습합니다.

In [ ]:
# Ch01: 신경망의 본질 — IEEE 14-bus MLP 전압 예측
# 환경 설정
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'pandapower', 'torch'])
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

# 재현성
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# 스타일
try:
    sys.path.insert(0, '..')
    from utils.style import set_style, COLORS
    set_style()
except:
    pass

## 1. 데이터 준비 — IEEE 14-bus 부하 시나리오 생성

pandapower로 IEEE 14-bus 시스템을 로드하고, 부하를 무작위로 변동시켜 학습 데이터를 생성합니다.
각 시나리오에서 부하 배율(0.5~1.5)을 입력, 조류 계산 결과의 버스 전압(p.u.)을 출력으로 사용합니다.

In [ ]:
import pandapower as pp
import pandapower.networks as pn

net = pn.case14()
base_p = net.load.p_mw.values.copy()
base_q = net.load.q_mvar.values.copy()

n_scenarios = 1000
load_factors = np.random.uniform(0.5, 1.5, (n_scenarios, len(base_p)))

X = []
Y = []

for i in range(n_scenarios):
    net.load.p_mw = base_p * load_factors[i]
    net.load.q_mvar = base_q * load_factors[i]
    try:
        pp.runpp(net, algorithm='nr', max_iteration=30)
        if net.converged:
            X.append(load_factors[i])
            Y.append(net.res_bus.vm_pu.values.copy())
    except:
        continue

X = np.array(X)
Y = np.array(Y)
print(f"유효 시나리오: {len(X)}개")

## 2. MLP 모델 구성 및 학습

3층 MLP를 구성합니다. 은닉층 64개 뉴런, ReLU 활성화 함수, Adam 옵티마이저, MSE 손실 함수를 사용합니다.

In [ ]:
class VoltagePredictor(nn.Module):
    def __init__(self, n_loads=11, n_buses=14, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_loads, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_buses),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
split = int(0.8 * len(X))
X_train, X_test = torch.FloatTensor(X[:split]), torch.FloatTensor(X[split:])
Y_train, Y_test = torch.FloatTensor(Y[:split]), torch.FloatTensor(Y[split:])

model = VoltagePredictor()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

losses = []
for epoch in range(500):
    optimizer.zero_grad()
    pred = model(X_train)
    loss = criterion(pred, Y_train)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.6f}")

## 3. 테스트 및 시각화

학습된 모델로 테스트 데이터의 전압을 예측하고, 실제 값과 비교합니다.

In [ ]:
with torch.no_grad():
    pred_test = model(X_test)
    test_loss = criterion(pred_test, Y_test)
    max_error = (pred_test - Y_test).abs().max()
    print(f"테스트 MSE: {test_loss.item():.6f}")
    print(f"최대 오차: {max_error.item():.4f} p.u.")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 실제 vs 예측
axes[0].scatter(Y_test.numpy().flatten(), pred_test.numpy().flatten(), alpha=0.3, s=5)
axes[0].plot([0.9, 1.1], [0.9, 1.1], 'r--')
axes[0].set_xlabel('실제 전압 (p.u.)')
axes[0].set_ylabel('예측 전압 (p.u.)')
axes[0].set_title('실제 vs 예측')

# 학습 손실 곡선
axes[1].plot(losses)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('학습 곡선')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()